In [ ]:
# !git clone https://github.com/vuthetam/RAG_Captioning.git

# import os
# import sys
# sys.path.append("/kaggle/working/RAG_Captioning")

In [ ]:
# %%writefile RAG_Captioning/main.py

# import os
# from pathlib import Path

# _defaults = {
#     "DATASET_COCO_PATH" : "/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json",
#     "IMAGES_PATH"        : "/kaggle/input/datasets/vuthetam/mscoco-2014/images",
#     "SAVE_LAST_CHECKPOINT_DIR": "/kaggle/working/checkpoints",
#     "SAVE_BEST_CHECKPOINT_DIR": "/kaggle/working/checkpoints",
#     "FREQ_THRESHOLD" : "5",
#     "DMODEL"         : "512",
#     "NHEADS"         : "8",
#     "NLAYERS"        : "4",
#     "BATCH_SIZE"     : "32",
#     "NUM_EPOCHS"     : "10",
#     "MAX_LENGTH"     : "40",
#     "LEARNING_RATE"  : "1e-4",
#     "WEIGHT_DECAY"   : "1e-4",
#     "MAX_GRAD_NORM"  : "1.0",
#     "NUM_WORKERS"    : "4",
#     "DROPOUT"        : "0.1",
# }
# os.environ.update(_defaults)


In [ ]:
# %%writefile -a RAG_Captioning/main.py

import sys
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from accelerate import Accelerator
from accelerate.utils import set_seed

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import (
    TRAIN_DF_PATH,
    VAL_DF_PATH,
    VOCAB_PATH,
    DMODEL,
    NHEADS,
    NLAYERS,
    BATCH_SIZE,
    NUM_EPOCHS,
    MAX_LENGTH,
    LEARNING_RATE,
    WEIGHT_DECAY,
    MAX_GRAD_NORM,
    NUM_WORKERS,
    DROPOUT,
    SAVE_LAST_CHECKPOINT_PATH,
    SAVE_BEST_CHECKPOINT_PATH,
    LOAD_LAST_CHECKPOINT_PATH,
)
from src.vocabulary import Vocabulary
from src.dataset import MSCOCODataset, create_clip_transform
from src.models.baseline import BaselineCaptioner
from src.engine import train_one_epoch, evaluate_one_epoch
from src.checkpoint import save_checkpoint, load_checkpoint
from src.utils import trainable_parameters

# Create Accelerator early so accelerator.print is available everywhere
accelerator = Accelerator(mixed_precision="fp16")
set_seed(42)

accelerator.print(f"PyTorch  : {torch.__version__}")
accelerator.print(f"CUDA     : {torch.cuda.is_available()}")
accelerator.print(f"Device   : {accelerator.device}")
accelerator.print(f"Mixed precision : {accelerator.mixed_precision}")

In [ ]:
# %%writefile -a RAG_Captioning/main.py

accelerator.print(
    f"epochs={NUM_EPOCHS}, lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY}, "
    f"batch={BATCH_SIZE}, max_len={MAX_LENGTH}, d_model={DMODEL}, "
    f"nheads={NHEADS}, nlayers={NLAYERS}, dropout={DROPOUT}"
)

In [ ]:
# %%writefile -a RAG_Captioning/main.py

train_df = pd.read_parquet(TRAIN_DF_PATH)
val_df   = pd.read_parquet(VAL_DF_PATH)

accelerator.print(f"train : {len(train_df):>7,} rows")
accelerator.print(f"val   : {len(val_df):>7,} rows")

vocab = Vocabulary.load(VOCAB_PATH)
accelerator.print(f"Vocabulary size : {len(vocab):,}")

In [ ]:
# %%writefile -a RAG_Captioning/main.py

transform = create_clip_transform()

train_dataset = MSCOCODataset(train_df, vocab, transform=transform, max_length=MAX_LENGTH)
val_dataset   = MSCOCODataset(val_df,   vocab, transform=transform, max_length=MAX_LENGTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

accelerator.print(f"train batches : {len(train_loader):,}")
accelerator.print(f"val   batches : {len(val_loader):,}")

In [ ]:
# %%writefile -a RAG_Captioning/main.py

model = BaselineCaptioner(
    vocab_size=len(vocab),
    d_model=DMODEL,
    nheads=NHEADS,
    nlayers=NLAYERS,
    dropout=DROPOUT,
    max_length=MAX_LENGTH,
    pad_idx=vocab.pad_idx(),
)

optimizer = torch.optim.AdamW(
    list(trainable_parameters(model)),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

trainable = sum(p.numel() for p in trainable_parameters(model))
total     = sum(p.numel() for p in model.parameters())
accelerator.print(f"Trainable params : {trainable:,} / {total:,}")

In [ ]:
# %%writefile -a RAG_Captioning/main.py

start_epoch   = 0
best_val_loss = float("inf")

if LOAD_LAST_CHECKPOINT_PATH and LOAD_LAST_CHECKPOINT_PATH.exists():
    start_epoch, last_train_loss, best_val_loss = load_checkpoint(
        LOAD_LAST_CHECKPOINT_PATH,
        model,
        optimizer,
        device=accelerator.device,
    )
    accelerator.print(f"Resumed after epoch {start_epoch}  |  best_val_loss={best_val_loss:.4f}")
else:
    accelerator.print("No checkpoint found – training from scratch.")

In [ ]:
# %%writefile -a RAG_Captioning/main.py

model, optimizer, train_loader, val_loader = accelerator.prepare(
    model, optimizer, train_loader, val_loader
)

In [ ]:
# %%writefile -a RAG_Captioning/main.py

pad_idx = vocab.pad_idx()

for epoch in range(start_epoch, NUM_EPOCHS):
    # Train
    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        pad_idx=pad_idx,
        accelerator=accelerator,
        max_grad_norm=MAX_GRAD_NORM,
        show_progress=True,
    )

    # Validate
    val_loss = evaluate_one_epoch(
        model=model,
        dataloader=val_loader,
        pad_idx=pad_idx,
        accelerator=accelerator,
        show_progress=True,
    )

    accelerator.print(f"[Epoch {epoch + 1:02d}/{NUM_EPOCHS}]  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    # Save last checkpoint — epoch+1 = number of epochs completed
    if SAVE_LAST_CHECKPOINT_PATH and accelerator.is_main_process:
        save_checkpoint(
            path=SAVE_LAST_CHECKPOINT_PATH,
            model=model,
            optimizer=optimizer,
            epoch=epoch + 1,
            train_loss=train_loss,
            best_val_loss=best_val_loss,
            accelerator=accelerator,
        )

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        if SAVE_BEST_CHECKPOINT_PATH and accelerator.is_main_process:
            save_checkpoint(
                path=SAVE_BEST_CHECKPOINT_PATH,
                model=model,
                optimizer=optimizer,
                epoch=epoch + 1,
                train_loss=train_loss,
                best_val_loss=best_val_loss,
                accelerator=accelerator,
            )

accelerator.print("\nTraining complete.")

In [ ]:
# !accelerate launch RAG_Captioning/main.py